# 信用卡违约预测：基于 Kaggle 数据集的机器学习分析

## 选题说明
本作业参考了 GitHub 仓库 **Default-Credit-Card-Prediction** 的分析思路，任务从数据理解、特征分析、类别不平衡处理、模型比较到结果解释，整体保持一致。  
与原仓库相同，本文选择 Kaggle 上的 **Default of Credit Card Clients Dataset** 作为实验数据集，它是台湾信用卡客户违约预测数据，包含 30,000 条样本、24 个属性，目标变量为 `default.payment.next.month`，属于二分类且类别不平衡问题。  

> 说明：如果课程要求“欺诈检测（fraud）”，可以把本 notebook 的数据集替换为 Kaggle 的 `Credit Card Fraud Detection`；但就与原仓库的“违约预测”任务匹配度而言，当前数据集是最一致的选择。

## 一、研究目标

1. 识别影响信用卡违约的关键因素。  
2. 构建能够预测下一月是否违约的分类模型。  
3. 比较不同预处理与采样策略对模型性能的影响。  
4. 以 F1-score、Recall、ROC-AUC 等指标作为主要评价标准，避免被不平衡数据下的 Accuracy 误导。

## 二、数据集简介

**数据集名称**：Default of Credit Card Clients Dataset  
**来源**：Kaggle（UCI 版本镜像）  
**样本量**：30,000  
**特征数**：24（含目标变量）  
**任务类型**：二分类预测  
**目标变量**：`default.payment.next.month`（1 = 违约，0 = 未违约）

特征主要分为四类：

- 客户基本信息：`LIMIT_BAL`, `SEX`, `EDUCATION`, `MARRIAGE`, `AGE`
- 还款状态：`PAY_0` ~ `PAY_6`
- 账单金额：`BILL_AMT1` ~ `BILL_AMT6`
- 还款金额：`PAY_AMT1` ~ `PAY_AMT6`

In [ ]:
# 基础库
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    precision_recall_curve, average_precision_score, f1_score,
    accuracy_score, recall_score, precision_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

plt.style.use("default")
sns.set_theme(style="whitegrid")

In [ ]:
# 1) 读取数据（UCI 与 Kaggle 版本字段一致）
# 若本地无 CSV，自动从 UCI 官方镜像下载（与 Kaggle 数据集相同）

WEEK1_DIR = Path(".").resolve()
DATA_PATH = WEEK1_DIR / "UCI_Credit_Card.csv"
FIG_DIR = WEEK1_DIR / "figures"
FIG_DIR.mkdir(exist_ok=True)

if not DATA_PATH.exists():
    from download_data import download
    download(DATA_PATH)

df = pd.read_csv(DATA_PATH)
print(f"Loaded {DATA_PATH.name}: {df.shape[0]:,} rows x {df.shape[1]} cols")
df.head()

In [ ]:
# 2) 基本信息检查
display(df.head())
print("Shape:", df.shape)
print("\nMissing values:")
print(df.isna().sum().sort_values(ascending=False).head(10))
print("\nTarget distribution:")
print(df["default.payment.next.month"].value_counts())
print(df["default.payment.next.month"].value_counts(normalize=True).round(4))

## 三、数据清洗与特征理解

原仓库已经指出：该数据集基本无缺失值，但存在两类需要注意的问题：

1. `ID` 对预测没有实际意义，可以删除；
2. `EDUCATION`、`MARRIAGE` 等分类编码需要先统一处理，再决定是否 one-hot 编码；
3. 类别不平衡明显，因此不能只看 Accuracy。

In [ ]:
# 3) 清洗
data = df.copy()

# 删除无预测意义的 ID
if "ID" in data.columns:
    data = data.drop(columns=["ID"])

# 一些常见的异常编码处理（可选）
# EDUCATION 中 0, 5, 6 常被视为“其他”
data["EDUCATION"] = data["EDUCATION"].replace({0: 4, 5: 4, 6: 4})
# MARRIAGE 中 0 常被视为“其他”
data["MARRIAGE"] = data["MARRIAGE"].replace({0: 3})

target = "default.payment.next.month"
X = data.drop(columns=[target])
y = data[target]

X.head()

In [ ]:
# 4) 简单 EDA
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(x=y, ax=axes[0])
axes[0].set_title("Target Distribution")
axes[0].set_xlabel("Default (0/1)")
axes[0].set_ylabel("Count")

sns.histplot(data["LIMIT_BAL"], bins=30, kde=True, ax=axes[1])
axes[1].set_title("Credit Limit Distribution")
axes[1].set_xlabel("LIMIT_BAL")

plt.tight_layout()
plt.show()

In [ ]:
# 5) 相关性热力图（只展示与目标相关较强的特征）
corr = data.corr(numeric_only=True)
top_corr = corr[target].abs().sort_values(ascending=False).head(12).index
plt.figure(figsize=(10, 8))
sns.heatmap(data[top_corr].corr(numeric_only=True), annot=False, cmap="coolwarm", center=0)
plt.title("Correlation Heatmap (Top related features)")
plt.show()

## 四、特征工程与数据划分

沿用原仓库的核心思路：

- 对数值特征进行标准化；
- 对分类特征进行 one-hot 编码；
- 训练集上采用 SMOTE 或随机下采样解决类别不平衡；
- 使用分层划分，保持训练/测试集类别比例一致。

In [ ]:
# 6) 划分特征类型
categorical_features = ["SEX", "EDUCATION", "MARRIAGE"]
numeric_features = [col for col in X.columns if col not in categorical_features]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

## 五、建模策略

这里选取与原仓库一致、并且适合课堂展示的三类模型：

1. **Logistic Regression**：作为可解释性强的基线模型；
2. **Random Forest**：捕获非线性关系；
3. **SVM**：在中小型数据上通常有较好表现。

每个模型都建议尝试两种不平衡处理方式：
- `SMOTE` 过采样；
- `RandomUnderSampler` 下采样。

评价时重点看：
- F1-score
- Recall
- ROC-AUC
- PR-AUC

In [ ]:
# 7) 模型定义
models = {
    "LogisticRegression": LogisticRegression(max_iter=2000, class_weight=None, random_state=42),
    "RandomForest": RandomForestClassifier(
        n_estimators=300, random_state=42, class_weight=None, n_jobs=-1
    ),
    "SVM": SVC(kernel="rbf", probability=True, random_state=42)
}

samplers = {
    "SMOTE": SMOTE(random_state=42),
    "RandomUnderSampler": RandomUnderSampler(random_state=42)
}

def build_pipeline(model, sampler=None):
    steps = [("preprocess", preprocess)]
    if sampler is not None:
        steps.append(("sampler", sampler))
    steps.append(("model", model))
    return ImbPipeline(steps)

In [ ]:
# 8) 交叉验证比较
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = []

for model_name, model in models.items():
    # 原始数据
    pipe = build_pipeline(model)
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    results.append({
        "model": model_name,
        "sampler": "None",
        **{k: np.mean(scores[f"test_{k}"]) for k in scoring}
    })

    # SMOTE
    pipe_smote = build_pipeline(model, samplers["SMOTE"])
    scores = cross_validate(pipe_smote, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    results.append({
        "model": model_name,
        "sampler": "SMOTE",
        **{k: np.mean(scores[f"test_{k}"]) for k in scoring}
    })

    # Under-sampling
    pipe_under = build_pipeline(model, samplers["RandomUnderSampler"])
    scores = cross_validate(pipe_under, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    results.append({
        "model": model_name,
        "sampler": "Under",
        **{k: np.mean(scores[f"test_{k}"]) for k in scoring}
    })

results_df = pd.DataFrame(results).sort_values(["f1", "roc_auc"], ascending=False)
results_df.to_csv(WEEK1_DIR / "model_comparison.csv", index=False)
print("Saved:", WEEK1_DIR / "model_comparison.csv")
display(results_df)

In [ ]:
# 9) 选择最佳方案并在测试集上评估
best_row = results_df.iloc[0]
best_model_name = best_row["model"]
best_sampler_name = best_row["sampler"]

best_model = models[best_model_name]
best_sampler = None if best_sampler_name == "None" else samplers["SMOTE"] if best_sampler_name == "SMOTE" else samplers["RandomUnderSampler"]

final_pipe = build_pipeline(best_model, best_sampler)
final_pipe.fit(X_train, y_train)

y_pred = final_pipe.predict(X_test)
y_prob = final_pipe.predict_proba(X_test)[:, 1] if hasattr(final_pipe, "predict_proba") else final_pipe.decision_function(X_test)

print("Best setting:", best_model_name, "+", best_sampler_name)
print("\nClassification report:\n")
print(classification_report(y_test, y_pred, digits=4))
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("Precision:", round(precision_score(y_test, y_pred), 4))
print("Recall:", round(recall_score(y_test, y_pred), 4))
print("F1:", round(f1_score(y_test, y_pred), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 4))

In [ ]:
# 10) 混淆矩阵
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.savefig(FIG_DIR / "03_confusion_matrix.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# 11) 画 ROC 曲线与 PR 曲线
from sklearn.metrics import roc_curve, precision_recall_curve, auc

fpr, tpr, _ = roc_curve(y_test, y_prob)
precision, recall, _ = precision_recall_curve(y_test, y_prob)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(fpr, tpr, label=f"ROC-AUC = {roc_auc_score(y_test, y_prob):.4f}")
axes[0].plot([0, 1], [0, 1], linestyle="--")
axes[0].set_title("ROC Curve")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].legend()

axes[1].plot(recall, precision, label=f"PR-AUC = {average_precision_score(y_test, y_prob):.4f}")
axes[1].set_title("Precision-Recall Curve")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].legend()

plt.tight_layout()
plt.savefig(FIG_DIR / "04_roc_pr_curves.png", dpi=120, bbox_inches="tight")
plt.show()

## 六、结果分析写作模板

运行 notebook 后，可以据此写入结论：

- 若 SMOTE 后 F1 和 Recall 更高，说明过采样更适合该数据集；
- 若 Random Forest 的 ROC-AUC 更好，说明非线性关系对违约预测更重要；
- 若 Logistic Regression 的解释性足够强，适合作为银行风控基线模型；
- 违约预测通常不能只看 Accuracy，因为“少数类”更重要。

### 可写入论文/报告的结论示例
> 在信用卡违约预测任务中，类别不平衡会明显影响模型学习效果。与直接使用原始数据相比，采用 SMOTE 进行过采样后，模型对违约样本的识别能力通常更强，F1-score 与 Recall 更具优势。综合考虑预测性能与可解释性，逻辑回归可作为基线模型，而随机森林适合进一步挖掘非线性关系。

## 七、参考文献

1. GitHub 仓库：Default-Credit-Card-Prediction  
2. Kaggle：Default of Credit Card Clients Dataset  
3. Yeh, I. C., & Lien, C. H. (2009). *The comparisons of data mining techniques for the predictive accuracy of probability of default of credit card clients*. Expert Systems with Applications, 36(2), 2473-2480.